In [2]:
import pandas as pd
import numpy as np
import pyodbc

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("pyodbc: installed ✅")
print("✅ All libraries ready!")

pandas: 2.3.3
numpy: 2.3.5
pyodbc: installed ✅
✅ All libraries ready!


In [3]:
# unoda actual file path pottunga
df = pd.read_csv(r"E:\DA-Portfilios\Project 3\web_analytics_fresh.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (12500, 14)

Columns: ['User_ID', 'Session_ID', 'Date', 'Traffic_Source', 'Device_Type', 'Browser', 'Country', 'Session_Duration', 'Pages_Visited', 'Bounce', 'Conversion', 'Engagement_Score', 'Engagement_Type', 'Bounce_Risk']

First 5 rows:


,User_ID,Session_ID,Date,Traffic_Source,Device_Type,Browser,Country,Session_Duration,Pages_Visited,Bounce,Conversion,Engagement_Score,Engagement_Type,Bounce_Risk
0,U05150,S05150,01-01-2022,Email,Mobile,Firefox,USA,278,6,Yes,No,147.3,Low Engaged,High Risk
1,U06008,S06008,01-01-2022,Google,Mobile,Chrome,Qatar,869,13,No,No,847.6,Highly Engaged,Low Risk
2,U06377,S06377,01-01-2022,Google,Desktop,Safari,UK,697,4,No,No,414.2,Moderately Engaged,Low Risk
3,U06741,S06741,01-01-2022,Twitter,Mobile,Safari,USA,1606,6,Yes,Yes,283.9,Low Engaged,High Risk
4,U07683,S07683,01-01-2022,Email,Mobile,Others,USA,40,13,No,No,525.5,Moderately Engaged,Low Risk


In [4]:
print("Data Types:")
print(df.dtypes)
print("\nNull Values:")
print(df.isnull().sum())
print("\nUnique Values:")
print(df.nunique())

Data Types:
User_ID              object
Session_ID           object
Date                 object
Traffic_Source       object
Device_Type          object
Browser              object
Country              object
Session_Duration      int64
Pages_Visited         int64
Bounce               object
Conversion           object
Engagement_Score    float64
Engagement_Type      object
Bounce_Risk          object
dtype: object

Null Values:
User_ID             0
Session_ID          0
Date                0
Traffic_Source      0
Device_Type         0
Browser             0
Country             0
Session_Duration    0
Pages_Visited       0
Bounce              0
Conversion          0
Engagement_Score    0
Engagement_Type     0
Bounce_Risk         0
dtype: int64

Unique Values:
User_ID             12500
Session_ID          12500
Date                 1491
Traffic_Source          8
Device_Type             3
Browser                 5
Country                12
Session_Duration     1789
Pages_Visited          

In [5]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

print("Date range:")
print("From:", df['Date'].min())
print("To:", df['Date'].max())
print("\n✅ Date format fixed!")

Date range:
From: 2022-01-01 00:00:00
To: 2026-01-31 00:00:00

✅ Date format fixed!


In [6]:
print("Engagement Type Distribution:")
print(df['Engagement_Type'].value_counts())

print("\nBounce Risk Distribution:")
print(df['Bounce_Risk'].value_counts())

print("\nDevice Type Distribution:")
print(df['Device_Type'].value_counts())

print("\nTraffic Source Distribution:")
print(df['Traffic_Source'].value_counts())

print("\n✅ Data validation done!")

Engagement Type Distribution:
Engagement_Type
Moderately Engaged    4720
Highly Engaged        4580
Low Engaged           3200
Name: count, dtype: int64

Bounce Risk Distribution:
Bounce_Risk
Low Risk     10544
High Risk     1956
Name: count, dtype: int64

Device Type Distribution:
Device_Type
Mobile     7183
Desktop    4575
Tablet      742
Name: count, dtype: int64

Traffic Source Distribution:
Traffic_Source
Google       3361
Direct       2244
Instagram    1974
Facebook     1825
Email        1091
Referral      895
Twitter       643
Others        467
Name: count, dtype: int64

✅ Data validation done!


In [7]:
# Connection string
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=LAPTOP-7MAV35VV\SQLEXPRESS;' 
    'DATABASE=master;'
    'Trusted_Connection=yes;'
)

cursor = conn.cursor()

# Database 
cursor.execute("""
    IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'Web_analytics')
    CREATE DATABASE Web_analytics
""")
conn.commit()
print("✅ Database created!")

✅ Database created!


In [8]:
cursor.execute("USE Web_analytics")

cursor.execute("""
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'web_sessions')
    CREATE TABLE web_sessions (
        User_ID VARCHAR(10),
        Session_ID VARCHAR(10),
        Date DATE,
        Traffic_Source VARCHAR(50),
        Device_Type VARCHAR(20),
        Browser VARCHAR(20),
        Country VARCHAR(50),
        Session_Duration INT,
        Pages_Visited INT,
        Bounce VARCHAR(5),
        Conversion VARCHAR(5),
        Engagement_Score FLOAT,
        Engagement_Type VARCHAR(30),
        Bounce_Risk VARCHAR(20)
    )
""")
conn.commit()
print("✅ Table created!")

✅ Table created!


In [12]:
conn2 = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=LAPTOP-7MAV35VV\\SQLEXPRESS;'
    'DATABASE=Web_analytics;'
    'Trusted_Connection=yes;'
)

cursor2 = conn2.cursor()

# Direct data insert
rows = [tuple(row) for row in df.itertuples(index=False)]
cursor2.executemany("""
    INSERT INTO web_sessions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
""", rows)

conn2.commit()
print(f"✅ {len(df)} rows inserted successfully!")

✅ 12500 rows inserted successfully!
